In [1]:
print("hi")

hi


In [6]:
import config
config.setup_logging()

from ingestion.pipeline import run_pipeline

store = run_pipeline(corpus_dir="corpus", out_dir="data")

12:30:06 [INFO] ingestion.pipeline: Starting ingestion pipeline (corpus=corpus, out=data)
12:30:06 [INFO] ingestion.loaders: Loaded 16 documents from corpus (0 skipped)
12:30:06 [INFO] ingestion.chunker: '99_unstructured_notes.md' matched no known format -- using fixed-size fallback (chunk_size=800, overlap=150)
12:30:06 [INFO] ingestion.pipeline: Chunked 16 documents into 49 chunks (1 used the fallback)
12:30:06 [INFO] ingestion.vector_store: Embedding 49 chunks and building the FAISS index...
12:30:07 [INFO] ingestion.vector_store: FAISS index built with 49 vectors
12:30:07 [INFO] ingestion.vector_store: Saved FAISS index to data
12:30:07 [INFO] ingestion.vector_store: Wrote human-readable chunk store to data\chunk_store.json
12:30:07 [INFO] ingestion.pipeline: Ingestion pipeline complete


INGESTION REPORT
Total documents: 16

Markdown documents: 11
PDF documents: 5

Total chunks: 49

Format 1 (reference_sentences) chunks: 12
Format 2 (key_points) chunks: 18
Format 3 (numbered_statements) chunks: 15
Fallback (fallback_fixed_size) chunks: 4

Documents that used the fallback (1):
  - 99_unstructured_notes.md  (did not match Format 1/2/3)

Sample chunks:
------------------------------------------------------------
chunk_id: '01_climate_001'
text: 'Scope 1 covers direct emissions; Scope 2 covers purchased energy; Scope 3 covers value-chain emissions.'
source: '01_climate.md'
title: 'Carbon Accounting Basics (Sample Corpus)'
file_type: 'md'
format: 'reference_sentences'
section: 'Reference Sentences'
page: None
chunk_index: 1
------------------------------------------------------------
chunk_id: '01_climate_002'
text: 'Activity data multiplied by emission factors yields estimated emissions.'
source: '01_climate.md'
title: 'Carbon Accounting Basics (Sample Corpus)'
file_type: 

In [7]:
from retrieval.tools import RetrievalTools

rt = RetrievalTools(data_dir="data")

query = "What does groundedness mean in evaluation?"
results = rt.search(query, top_k=5)
results

12:30:18 [INFO] ingestion.vector_store: Loaded FAISS index from data (49 vectors)
12:30:18 [INFO] retrieval.keyword: Loaded 49 chunks for BM25 from data\chunk_store.json
12:30:18 [INFO] retrieval.tools: RetrievalTools ready (49 chunks, USE_BM25=True, USE_RERANKER=True)
Batches: 100%|██████████| 1/1 [00:00<00:00,  3.51it/s]
12:30:18 [INFO] retrieval.tools: search('What does groundedness mean in evaluation?') -> 5 results in 317.2ms


[{'chunk_id': '22_ai_evals_002',
  'source': '22_ai_evals.md',
  'section': 'Key Points',
  'page': None,
  'score': 0.185592383146286},
 {'chunk_id': 'policy_eval_playbook_001',
  'source': 'policy_eval_playbook.pdf',
  'section': None,
  'page': 1,
  'score': -4.3877763748168945},
 {'chunk_id': '22_ai_evals_001',
  'source': '22_ai_evals.md',
  'section': 'Key Points',
  'page': None,
  'score': -6.141379356384277},
 {'chunk_id': '01_ops_incidents_002',
  'source': '01_ops_incidents.md',
  'section': 'Reference Sentences',
  'page': None,
  'score': -10.954463958740234},
 {'chunk_id': 'guide_evidence_citations_002',
  'source': 'guide_evidence_citations.pdf',
  'section': None,
  'page': 1,
  'score': -11.010360717773438}]

In [ ]:
rt.read_chunk(results[0]["chunk_id"])


'Evaluation should separate retrieval quality from generation quality.'

In [16]:
QUERY = "In RAG pipeline what does reranking do?"

def run_config(use_bm25, use_reranker, label):
    config.USE_BM25 = use_bm25
    config.USE_RERANKER = use_reranker
    rt = RetrievalTools(data_dir="data")
    results = rt.search(QUERY, top_k=3)
    print(f"\n=== {label} ===")
    for r in results:
        text = rt.read_chunk(r["chunk_id"])
        print(r)
        print(f"  text: {text}\n")

run_config(False, False, "Semantic only")
run_config(True, False, "Semantic + BM25 + RRF")
run_config(True, True, "Semantic + BM25 + RRF + Reranker")

12:50:53 [INFO] ingestion.vector_store: Loaded FAISS index from data (49 vectors)
12:50:53 [INFO] retrieval.tools: RetrievalTools ready (49 chunks, USE_BM25=False, USE_RERANKER=False)
12:50:53 [INFO] retrieval.tools: search('In RAG pipeline what does reranking do?') -> 3 results in 24.3ms
12:50:53 [INFO] ingestion.vector_store: Loaded FAISS index from data (49 vectors)
12:50:53 [INFO] retrieval.keyword: Loaded 49 chunks for BM25 from data\chunk_store.json
12:50:53 [INFO] retrieval.tools: RetrievalTools ready (49 chunks, USE_BM25=True, USE_RERANKER=False)
12:50:53 [INFO] retrieval.tools: search('In RAG pipeline what does reranking do?') -> 3 results in 16.5ms
12:50:53 [INFO] ingestion.vector_store: Loaded FAISS index from data (49 vectors)
12:50:53 [INFO] retrieval.keyword: Loaded 49 chunks for BM25 from data\chunk_store.json
12:50:53 [INFO] retrieval.tools: RetrievalTools ready (49 chunks, USE_BM25=True, USE_RERANKER=True)



=== Semantic only ===
{'chunk_id': '12_rag_003', 'source': '12_rag.md', 'section': 'Key Points', 'page': None, 'score': 0.47741711139678955}
  text: A reranker can improve precision by reordering retrieved chunks.

{'chunk_id': '12_rag_001', 'source': '12_rag.md', 'section': 'Key Points', 'page': None, 'score': 0.44714266061782837}
  text: RAG typically includes ingestion, chunking, embedding, indexing, retrieval, and synthesis.

{'chunk_id': 'guide_vector_index_003', 'source': 'guide_vector_index.pdf', 'section': None, 'page': 1, 'score': 0.3503884971141815}
  text: Reranking can improve top-k precision for question answering.


=== Semantic + BM25 + RRF ===
{'chunk_id': '12_rag_001', 'source': '12_rag.md', 'section': 'Key Points', 'page': None, 'score': 0.03252247488101534}
  text: RAG typically includes ingestion, chunking, embedding, indexing, retrieval, and synthesis.

{'chunk_id': '07_prompt_injection_003', 'source': '07_prompt_injection.md', 'section': 'Key Points', 'page': Non

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.81it/s]
12:50:53 [INFO] retrieval.tools: search('In RAG pipeline what does reranking do?') -> 3 results in 230.9ms



=== Semantic + BM25 + RRF + Reranker ===
{'chunk_id': '12_rag_003', 'source': '12_rag.md', 'section': 'Key Points', 'page': None, 'score': -0.7934340238571167}
  text: A reranker can improve precision by reordering retrieved chunks.

{'chunk_id': 'guide_vector_index_003', 'source': 'guide_vector_index.pdf', 'section': None, 'page': 1, 'score': -3.005197525024414}
  text: Reranking can improve top-k precision for question answering.

{'chunk_id': '12_rag_001', 'source': '12_rag.md', 'section': 'Key Points', 'page': None, 'score': -6.530288219451904}
  text: RAG typically includes ingestion, chunking, embedding, indexing, retrieval, and synthesis.



In [1]:
import config
config.setup_logging()

from agent.react_agent import build_agent, run_agent

agent = build_agent()
result = run_agent("What does 'faithfulness' mean in LLM evaluation?", agent=agent)

print(result["answer"])
print("\nCitations valid:", result["citations_valid"], result["citation_problems"])

e:\react_copilot_ingestion_3\react_copilot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
10:55:43 [INFO] ingestion.embeddings: Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
10:55:44 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
10:55:44 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
10:55:44 [INFO] httpx: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_

In the context of LLM (Large Language Model) evaluation, "faithfulness" refers to the extent to which an answer is supported by retrieved evidence. This means that the information provided by the model should be verifiable and aligned with the sources it references. Specifically, faithfulness ensures that the model's outputs are not only accurate but also grounded in reliable data or evidence that can be traced back to the original sources [policy_eval_playbook.pdf:1] "the answer is supported by retrieved evidence."

Citations valid: True []


In [2]:
for step in result["trace"]:
    print(f"\nStep {step['step']}: {step['action']}({step['action_input']})")
    print(f"  Observation: {step['observation']}")


Step 1: search({'query': 'faithfulness LLM evaluation'})
  Observation: [{"chunk_id": "policy_eval_playbook_001", "source": "policy_eval_playbook.pdf", "section": null, "page": 1, "score": -1.6477415561676025}, {"chunk_id": "22_ai_evals_002", "source": "22_ai_evals.md", "section": "Key Points", "page": null, "score": -4.544339179992676}, {"chunk_id": "22_ai_evals_001", "source": "22_ai_evals.md", "section": "Key Points", "page": null, "score": -10.667215347290039}, {"chunk_id": "12_rag_002", "source": "12_rag.md", "section": "Key Points", "page": null, "score": -11.36750602722168}, {"chunk_id": "policy_safety_notes_002", "source": "policy_safety_notes.pdf", "section": null, "page": 1, "score": -11.381641387939453}]

Step 2: search({'query': 'faithfulness definition LLM evaluation'})
  Observation: [{"chunk_id": "policy_eval_playbook_001", "source": "policy_eval_playbook.pdf", "section": null, "page": 1, "score": 2.5657553672790527}, {"chunk_id": "22_ai_evals_002", "source": "22_ai_eva

In [4]:
import config
config.setup_logging()

from agent.react_agent import build_agent, run_agent

agent = build_agent()
result = run_agent("why do agent systems use max-steps limit?", agent=agent)

print(result["answer"])
print("\nCitations valid:", result["citations_valid"], result["citation_problems"])

11:16:24 [INFO] ingestion.vector_store: Loaded FAISS index from data (49 vectors)
11:16:24 [INFO] retrieval.keyword: Loaded 49 chunks for BM25 from data\chunk_store.json
11:16:24 [INFO] retrieval.tools: RetrievalTools ready (49 chunks, USE_BM25=True, USE_RERANKER=True)
11:16:24 [INFO] agent.tools: Built 2 tool(s) from allow-list: ['search', 'read_chunk']
11:16:24 [INFO] agent.react_agent: ReAct agent built with 2 tool(s)
11:16:29 [INFO] httpx2: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00,  5.94it/s]
11:16:29 [INFO] retrieval.tools: search('agent systems max-steps limit') -> 5 results in 557.5ms
11:16:29 [INFO] agent.tools: [tool] search('agent systems max-steps limit') -> 5 results
11:16:30 [INFO] httpx2: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
11:16:30 [INFO] agent.tools: [tool] read_chunk('policy_eval_playbook_003') -> 61 chars
11:16:33 [INFO] httpx2: HTTP Request: P

Agent systems use a max-steps limit to prevent infinite loops during execution. This safeguard ensures that the system does not get stuck in a cycle of actions without reaching a conclusion or a desired state. [policy_eval_playbook.pdf:1] "A max-steps limit prevents infinite loops in agent execution."

Citations valid: True []
